# #204 Track A — outcome-unexposed Gate A

This executed notebook is deliberately bounded to the preregistration. It verifies the immutable #203 inference source and the #204 split/model/fallback contract, but never imports a settlement-label module or opens 2024 test outcomes.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd()
PACKAGE = ROOT / 'results/bj204_gate_a_preoutcome_v1'
assert PACKAGE.is_dir(), PACKAGE
assert 'unified_tensor.settlement_labels' not in sys.modules
print('Gate-A package:', PACKAGE)

Gate-A package: /home/drwho1985/workspace/soccer/beijing-lot/.worktrees/strategy/.scratch/bj204_prob_sp/results/bj204_gate_a_preoutcome_v1


In [2]:
from unified_model.gate_a import verify_package

verification = verify_package(PACKAGE)
verification

{'contract_version': 'bj204-market-anchored-model-gate-a-v1',
 'files': 8,
 'status': 'PASS'}

In [3]:
splits = pd.read_csv(PACKAGE / 'split_manifest.csv', dtype={'issue': str})
summary = (splits.groupby('role', sort=True)
           .agg(issues=('issue_instance', 'size'), first_year=('year', 'min'), last_year=('year', 'max'))
           .reset_index())
assert not splits['labels_permitted_gate_a'].astype(str).str.lower().eq('true').any()
summary

,role,issues,first_year,last_year
0,DEVELOPMENT,270,2019,2023
1,EMBARGOED_FOR_210,65,2025,2026
2,RETROSPECTIVE_TEST_LOCKED,59,2024,2024


In [4]:
card = json.loads((PACKAGE / 'freeze_card.json').read_text(encoding='utf-8'))
{
    'source_labels_opened': card['source']['labels_opened'],
    'development_years': card['chronology']['development_years'],
    'retrospective_test_year': card['chronology']['retrospective_test_year'],
    'embargoed_start_year': card['chronology']['embargoed_start_year'],
    'fallback': card['fallback']['rule_id'],
    'models': [row['id'] for row in card['models']],
}

{'source_labels_opened': False,
 'development_years': [2019, 2020, 2021, 2022, 2023],
 'retrospective_test_year': 2024,
 'embargoed_start_year': 2025,
 'fallback': 'BJ204_MARKET_OR_NO_BET_V1',
 'models': ['MARKET_ONLY',
  'LOGISTIC_RESIDUAL',
  'HIST_GRADIENT_RESIDUAL',
  'SET_ENCODER_RESIDUAL']}